# Network Flow V2 — Edge-IIoTset Dataset Audit

**Dataset:** `ML-EdgeIIoT-dataset.csv` (selected ML subset, not necessarily the complete original Edge-IIoTset release)  
**Purpose:** inspect the *actual local file* before cleaning, feature selection, splitting, or modelling.

This notebook reads the CSV in chunks for full-file statistics, without changing the original data. It writes a small audit JSON report to `reports/audits`. **No model training occurs here.**

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd
from IPython.display import display

ROOT = Path(r"D:\J26-CS-336-Network-Flow-V2\network_flow_v2")
DATA = ROOT / "data" / "raw" / "edge_iiotset" / "ML-EdgeIIoT-dataset.csv"
OUT = ROOT / "reports" / "audits" / "edge_iiotset_audit_v1.json"

assert DATA.is_file(), f"Dataset not found: {DATA}"
print("Dataset:", DATA)
print("File size (MB):", round(DATA.stat().st_size / (1024 ** 2), 2))
print("Exists:", DATA.is_file())

## 1. Inspect the schema and a small sample

This reads only **1,000 rows** to identify columns and likely labels, not to estimate final dataset statistics.

In [ ]:
sample = pd.read_csv(DATA, nrows=1000, low_memory=False)
columns = sample.columns.tolist()
print(f"Columns: {len(columns)}")
print("Column names:")
for i, col in enumerate(columns, start=1):
    print(f"{i:>3}. {col}  [{sample[col].dtype}]")

print("\nFirst 3 observations (displayed for inspection only):")
display(sample.head(3))

## 2. Find candidate ground-truth columns

Ground truth is the known correct label used to evaluate the classifier. The labels must **never** become model input features. We only identify candidate label columns here; their meaning still needs verification.

In [ ]:
def normalized_column_name(value):
    return str(value).strip().lower().replace(" ", "").replace("_", "").replace("-", "")

label_names = {
    "attacklabel", "attacktype", "label", "class", "binarylabel",
    "category", "attackcategory", "attack"
}
label_columns = [c for c in columns if normalized_column_name(c) in label_names]
print("Possible label columns:", label_columns)
if not label_columns:
    print("No obvious label column found; inspect the schema above before continuing.")

## 3. Full-file audit (chunked)

Counts **all rows** in the local CSV, calculates CSV/pandas-recognized missing values, and tallies candidate labels. It does **not** drop duplicates, replace missing values, normalize features, or silently correct any labels. Empty strings and placeholder values such as `?` may require an additional audit.

In [ ]:
row_count = 0
null_counts = Counter()
label_counts = {c: Counter() for c in label_columns}
chunk_size = 50_000

for chunk_number, chunk in enumerate(
    pd.read_csv(DATA, chunksize=chunk_size, low_memory=False), start=1
):
    row_count += len(chunk)
    null_counts.update({c: int(v) for c, v in chunk.isna().sum().items()})
    for col in label_columns:
        # Keep raw label spellings, strip whitespace, distinguish missing explicitly.
        cleaned = chunk[col].astype("string").str.strip().fillna("<MISSING>")
        label_counts[col].update({str(k): int(v) for k, v in cleaned.value_counts().items()})
    print(f"Chunk {chunk_number}: {row_count:,} rows read", end="\r")

print(f"\nFULL AUDIT COMPLETE: {row_count:,} records; {len(columns)} columns")
missing_table = pd.DataFrame([
    {"feature": c, "missing_count": int(null_counts[c]),
     "missing_pct": round(100 * null_counts[c] / row_count, 4) if row_count else 0.0}
    for c in columns
]).sort_values("missing_count", ascending=False)
print("\nMost missing fields:")
display(missing_table.head(15))

## 4. Label distribution

Counts are specific to **your downloaded `ML-EdgeIIoT-dataset.csv`**, which may have a different class ratio from figures published for the full Edge-IIoTset release.

In [ ]:
if not label_columns:
    print("STOP: No candidate label column detected. Please share the column names.")
else:
    for col in label_columns:
        print(f"\nLabel column: {col}")
        label_table = pd.DataFrame(
            [{"label": key, "count": count,
              "percentage": round(100 * count / row_count, 6) if row_count else 0.0}
             for key, count in label_counts[col].most_common()]
        )
        display(label_table)
        print("Total in this label column:", int(label_table["count"].sum()))

## 5. Save reproducible audit report

A SHA-256 file checksum helps identify the exact CSV version. The resulting JSON contains counts and metadata, not the underlying dataset records.

In [ ]:
digest = hashlib.sha256()
with DATA.open("rb") as handle:
    for block in iter(lambda: handle.read(1024 * 1024), b""):
        digest.update(block)

report = {
    "dataset": "Edge-IIoTset selected ML CSV",
    "source_filename": DATA.name,
    "size_bytes": DATA.stat().st_size,
    "sha256": digest.hexdigest(),
    "audited_utc": datetime.now(timezone.utc).isoformat(),
    "row_count": row_count,
    "column_count": len(columns),
    "columns": columns,
    "candidate_label_columns": label_columns,
    "raw_label_distributions": {c: dict(label_counts[c]) for c in label_columns},
    "pandas_missing_counts": {c: int(null_counts[c]) for c in columns},
    "caveats": [
        "Attack label meanings and class mapping require documentation review.",
        "Only pandas-recognized missing values counted; custom placeholders not yet audited.",
        "Duplicates, label leakage, capture/session identities, and group/time split feasibility not yet audited.",
        "Published full-dataset class distributions may differ from this selected ML CSV."
    ]
}
OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved audit report:", OUT)
print("SHA-256:", report["sha256"])
print("Records:", report["row_count"])
print("Candidate labels:", report["candidate_label_columns"])

## Next step — do not split yet

Send the following **actual notebook outputs** for review:
1. Number of records and columns.
2. Full list of column names (or copy the printed output).
3. `Attack_label` / `Attack_type` or other candidate-label distribution tables.
4. Top missing-value fields.

After confirming the label definitions and available session/time/group identifiers, design a **70% train / 15% validation / 15% test** split and check for duplicates and leakage. Splitting should respect campaign/capture/device grouping where those identifiers are available, not automatically use random rows.